In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import ndimage


# ============================================
# INPUT FILES
# ============================================
NOTEBOOK_DIR = Path("/Users/zd3504phd/Desktop/XAIV/analysis/imagenet_mask_vis")
RESULTS_CSV = Path("/Users/zd3504phd/Desktop/XAIV/benchmarks/vggnet16_benchmark2022_segmented_all_imgs/debug_vis/property_masks.csv")
FILTER_CSV = Path("/Users/zd3504phd/Desktop/XAIV/results/vggnet16/ALL_vggnet.csv")
OUTPUT_CSV = NOTEBOOK_DIR / "filtered_flip_results.csv"
MASK_BASE_DIR = Path("/Users/zd3504phd/Desktop/XAIV/benchmarks/vggnet16_benchmark2022_segmented_all_imgs/debug_vis")


# ============================================
# LOAD CSVs
# ============================================
df_results = pd.read_csv(RESULTS_CSV)
df_filter = pd.read_csv(FILTER_CSV)

df_results.columns = df_results.columns.str.strip()
df_filter.columns = df_filter.columns.str.strip()


# ============================================
# FILTER TO THE IMAGES WE WANT TO EVALUATE
# ============================================
filter_images = (
    df_filter[["image"]]
    .dropna(subset=["image"])
    .drop_duplicates()
    .reset_index(drop=True)
)
print(f"Filtering to {len(filter_images)} unique images from {FILTER_CSV.name}")

df_fix_mask = (
    filter_images.merge(
        df_results[df_results["tag"] == "fix_mask"].copy(),
        on="image",
        how="inner",
        validate="one_to_one",
    )
    .copy()
)

df_fix_mask["imagenet_id"] = pd.to_numeric(df_fix_mask["top1_label"], errors="coerce").astype("Int64")

missing_fix_mask_images = sorted(set(filter_images["image"]) - set(df_fix_mask["image"]))
print(f"Matched {len(df_fix_mask)} fix_mask rows")
print(f"Images in filter CSV without a fix_mask row: {len(missing_fix_mask_images)}")
print(f"Rows missing imagenet_id labels: {int(df_fix_mask['imagenet_id'].isna().sum())}")


# ============================================
# HELPERS
# ============================================
def resolve_mask_path(mask_path: str, base_dir: Path) -> Path:
    path = Path(mask_path)
    if path.is_absolute():
        return path
    return base_dir / path


def load_mask_from_npz(npz_path: Path) -> np.ndarray:
    data = np.load(npz_path, allow_pickle=False)

    if isinstance(data, np.lib.npyio.NpzFile):
        preferred_keys = ["mask", "arr_0", "masks", "segmentation"]
        key = next((candidate for candidate in preferred_keys if candidate in data.files), None)
        if key is None:
            if len(data.files) == 0:
                raise ValueError(f"No arrays found in {npz_path}")
            key = data.files[0]
        arr = np.asarray(data[key])
    else:
        arr = np.asarray(data)

    arr = np.squeeze(arr)
    if arr.ndim != 2:
        raise ValueError(f"Expected a 2D mask in {npz_path}, got shape {arr.shape}")

    return arr.astype(bool)


def compute_largest_cc_fraction(mask: np.ndarray) -> float:
    mask_area = int(mask.sum())
    if mask_area == 0:
        return 0.0

    labeled, num = ndimage.label(mask)
    if num == 0:
        return 0.0

    sizes = ndimage.sum(mask, labeled, index=np.arange(1, num + 1))
    largest = float(np.max(sizes)) if len(sizes) > 0 else 0.0
    return largest / mask_area


def compute_center_occupancy(mask: np.ndarray) -> float:
    h, w = mask.shape
    mask_area = int(mask.sum())
    if mask_area == 0:
        return 0.0

    y1 = h // 4
    y2 = 3 * h // 4
    x1 = w // 4
    x2 = 3 * w // 4

    return float(mask[y1:y2, x1:x2].sum()) / mask_area


def compute_outer_band_fraction(mask: np.ndarray, band_fraction: float = 0.125) -> float:
    h, w = mask.shape
    mask_area = int(mask.sum())
    if mask_area == 0:
        return 0.0

    band = max(1, int(round(min(h, w) * band_fraction)))
    outer = np.zeros_like(mask, dtype=bool)
    outer[:band, :] = True
    outer[-band:, :] = True
    outer[:, :band] = True
    outer[:, -band:] = True

    return float((mask & outer).sum()) / mask_area


def compute_features(mask: np.ndarray, fallback_r=None) -> dict:
    total_pixels = mask.size
    mask_pixels = int(mask.sum())

    r = 0.0 if total_pixels == 0 else mask_pixels / total_pixels
    if fallback_r is not None and pd.notna(fallback_r):
        r = float(fallback_r)

    lam = compute_largest_cc_fraction(mask)
    c = compute_center_occupancy(mask)
    outer_frac = compute_outer_band_fraction(mask)

    return {
        "r": r,
        "lambda": lam,
        "c": c,
        "outer_frac": outer_frac,
    }


def classify_flipped(features: dict) -> dict:
    r = features["r"]
    lam = features["lambda"]
    c = features["c"]
    outer_frac = features["outer_frac"]

    small_mask_threshold = 0.08
    large_mask_threshold = 0.401905
    outer_band_threshold = 0.711311
    empty_center_threshold = 0.01436

    central_compact_thresholds = {
        "lambda": 0.80,
        "c": 0.58,
        "outer_frac": 0.15,
        "r": 0.17,
    }
    high_connectivity_thresholds = {
        "lambda": 0.85,
        "r": 0.30,
        "outer_frac": 0.20,
        "c": 0.50,
    }

    rule_margins = {
        "large_mask_ratio": r - large_mask_threshold,
        "outer_band_dominant": outer_frac - outer_band_threshold,
        "empty_center": empty_center_threshold - c,
        "central_compact_object": min(
            lam - central_compact_thresholds["lambda"],
            c - central_compact_thresholds["c"],
            central_compact_thresholds["outer_frac"] - outer_frac,
            r - central_compact_thresholds["r"],
        ),
        "high_connectivity_midlarge": min(
            lam - high_connectivity_thresholds["lambda"],
            r - high_connectivity_thresholds["r"],
            outer_frac - high_connectivity_thresholds["outer_frac"],
            high_connectivity_thresholds["c"] - c,
        ),
    }
    flip_score = max(rule_margins.values())

    if r < small_mask_threshold:
        return {
            "is_flipped": False,
            "decision_rule": "small_mask_guard",
            "flip_score": flip_score,
        }

    if rule_margins["large_mask_ratio"] >= 0.0:
        return {
            "is_flipped": True,
            "decision_rule": "large_mask_ratio",
            "flip_score": flip_score,
        }

    if rule_margins["outer_band_dominant"] >= 0.0:
        return {
            "is_flipped": True,
            "decision_rule": "outer_band_dominant",
            "flip_score": flip_score,
        }

    if rule_margins["empty_center"] >= 0.0:
        return {
            "is_flipped": True,
            "decision_rule": "empty_center",
            "flip_score": flip_score,
        }

    if rule_margins["central_compact_object"] >= 0.0:
        return {
            "is_flipped": True,
            "decision_rule": "central_compact_object",
            "flip_score": flip_score,
        }

    if rule_margins["high_connectivity_midlarge"] >= 0.0:
        return {
            "is_flipped": True,
            "decision_rule": "high_connectivity_midlarge",
            "flip_score": flip_score,
        }

    return {
        "is_flipped": False,
        "decision_rule": "default",
        "flip_score": flip_score,
    }


# ============================================
# RUN FLIP DETECTION ON THE FILTERED fix_mask ROWS
# ============================================
rows = []

for _, row in df_fix_mask.iterrows():
    image = row["image"]
    seg = row["segment_index"]
    mask_path = row["mask_path"]
    full_mask_path = resolve_mask_path(mask_path, MASK_BASE_DIR)

    base_record = {
        "image": image,
        "imagenet_id": int(row["imagenet_id"]) if pd.notna(row["imagenet_id"]) else pd.NA,
        "segment_index": seg,
        "tag": row["tag"],
        "mask_path": mask_path,
        "resolved_mask_path": str(full_mask_path),
    }

    if not full_mask_path.exists():
        rows.append({
            **base_record,
            "status": "missing_mask_file",
            "r": np.nan,
            "lambda": np.nan,
            "c": np.nan,
            "flip_score": np.nan,
            "is_flipped": np.nan,
            "decision_rule": "missing_mask_file",
        })
        continue

    try:
        mask = load_mask_from_npz(full_mask_path)
        fallback_r = row["mask_true_fraction"] if "mask_true_fraction" in row.index else None

        feats = compute_features(mask, fallback_r=fallback_r)
        pred = classify_flipped(feats)

        rows.append({
            **base_record,
            "status": "ok",
            "r": feats["r"],
            "lambda": feats["lambda"],
            "c": feats["c"],
            "outer_frac": feats["outer_frac"],
            "flip_score": pred["flip_score"],
            "is_flipped": pred["is_flipped"],
            "decision_rule": pred["decision_rule"],
        })

    except Exception as exc:
        rows.append({
            **base_record,
            "status": f"error: {exc}",
            "r": np.nan,
            "lambda": np.nan,
            "c": np.nan,
            "outer_frac": np.nan,
            "flip_score": np.nan,
            "is_flipped": np.nan,
            "decision_rule": "error",
        })

df_flip = pd.DataFrame(rows)

print(f"Processed {len(df_flip)} masks")
print(df_flip.columns.tolist())
print(df_flip["status"].value_counts(dropna=False))

if "is_flipped" in df_flip.columns:
    print(df_flip["is_flipped"].value_counts(dropna=False))

df_flip.to_csv(OUTPUT_CSV, index=False)
print(f"Saved to {OUTPUT_CSV}")

Filtering to 182 unique images from ALL_vggnet.csv
Matched 182 fix_mask rows
Images in filter CSV without a fix_mask row: 0
Rows missing imagenet_id labels: 0


Processed 182 masks
['image', 'imagenet_id', 'segment_index', 'tag', 'mask_path', 'resolved_mask_path', 'status', 'r', 'lambda', 'c', 'outer_frac', 'flip_score', 'is_flipped', 'decision_rule']
status
ok    182
Name: count, dtype: int64
is_flipped
False    98
True     84
Name: count, dtype: int64
Saved to /Users/zd3504phd/Desktop/XAIV/analysis/imagenet_mask_vis/filtered_flip_results.csv


In [2]:
import pandas as pd


# --------------------------------------------
# Stable ground-truth labels in ImageNet ID space
# --------------------------------------------
flipped_imagenet_ids = {
    16, 18, 30, 35, 36, 61, 63, 68, 76, 87, 101, 102, 105, 108, 111, 114, 117,
    119, 120, 121, 127, 133, 134, 151, 152, 155, 162, 164, 165, 169, 170, 171,
    173, 201, 202, 207, 210, 211, 212, 229, 234, 252, 253, 254, 255, 259, 260,
    263, 271,
}
uncertain_imagenet_ids = {88, 91, 100, 132, 233}


# --------------------------------------------
# df_flip must already exist and contain:
# imagenet_id, r, lambda, c, outer_frac, flip_score, is_flipped, status
# --------------------------------------------
eval_df = df_flip[df_flip["status"] == "ok"].copy()
eval_df = eval_df.dropna(subset=["imagenet_id"]).copy()
eval_df["imagenet_id"] = eval_df["imagenet_id"].astype(int)

evaluable_imagenet_ids = sorted(eval_df["imagenet_id"].unique())
print(f"Evaluable filtered images: {len(evaluable_imagenet_ids)}")
print(f"Flipped ground-truth IDs: {len(flipped_imagenet_ids)}")
print(f"Uncertain ground-truth IDs: {len(uncertain_imagenet_ids)}")

missing_flipped_ids = sorted(flipped_imagenet_ids - set(evaluable_imagenet_ids))
missing_uncertain_ids = sorted(uncertain_imagenet_ids - set(evaluable_imagenet_ids))
print(f"Missing flipped IDs in this filtered run: {missing_flipped_ids}")
print(f"Missing uncertain IDs in this filtered run: {missing_uncertain_ids}")

eval_df = eval_df[~eval_df["imagenet_id"].isin(uncertain_imagenet_ids)].copy()
eval_df["y"] = eval_df["imagenet_id"].isin(flipped_imagenet_ids).astype(int)
print(f"Images used for metric evaluation: {len(eval_df)}")
print("\nCurrent rule metric:")
print("  small_mask_guard: r < 0.08 -> not flipped")
print("  large_mask_ratio: r >= 0.401905 -> flipped")
print("  outer_band_dominant: outer_frac >= 0.711311 -> flipped")
print("  empty_center: c <= 0.01436 -> flipped")
print("  central_compact_object: lambda >= 0.80 and c >= 0.58 and outer_frac <= 0.15 and r >= 0.17 -> flipped")
print("  high_connectivity_midlarge: lambda >= 0.85 and r >= 0.30 and outer_frac >= 0.20 and c <= 0.50 -> flipped")

y = eval_df["y"].to_numpy()
pred = eval_df["is_flipped"].astype(bool).astype(int).to_numpy()
eval_df["pred"] = pred
tn = int(((y == 0) & (pred == 0)).sum())
fp = int(((y == 0) & (pred == 1)).sum())
fn = int(((y == 1) & (pred == 0)).sum())
tp = int(((y == 1) & (pred == 1)).sum())
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = 2.0 * precision * recall / (precision + recall) if (precision + recall) else 0.0
balanced_accuracy = 0.5 * ((tp / (tp + fn)) + (tn / (tn + fp)))
print()
print(f"TP: {tp}")
print(f"FN: {fn}")
print(f"FP: {fp}")
print(f"TN: {tn}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1: {f1:.4f}")
print(f"Balanced accuracy: {balanced_accuracy:.4f}")

# show mistakes
print("\nFalse negatives:")
print(eval_df[(eval_df["y"] == 1) & (eval_df["pred"] == 0)][
    ["imagenet_id", "image", "r", "lambda", "c", "outer_frac", "flip_score", "decision_rule"]
].sort_values("imagenet_id"))

print("\nFalse positives:")
print(eval_df[(eval_df["y"] == 0) & (eval_df["pred"] == 1)][
    ["imagenet_id", "image", "r", "lambda", "c", "outer_frac", "flip_score", "decision_rule"]
].sort_values("imagenet_id"))

Evaluable filtered images: 182
Flipped ground-truth IDs: 49
Uncertain ground-truth IDs: 5
Missing flipped IDs in this filtered run: []
Missing uncertain IDs in this filtered run: []
Images used for metric evaluation: 177

Current rule metric:
  small_mask_guard: r < 0.08 -> not flipped
  large_mask_ratio: r >= 0.401905 -> flipped
  outer_band_dominant: outer_frac >= 0.711311 -> flipped
  empty_center: c <= 0.01436 -> flipped
  central_compact_object: lambda >= 0.80 and c >= 0.58 and outer_frac <= 0.15 and r >= 0.17 -> flipped
  high_connectivity_midlarge: lambda >= 0.85 and r >= 0.30 and outer_frac >= 0.20 and c <= 0.50 -> flipped

TP: 42
FN: 7
FP: 39
TN: 89
Precision: 0.5185
Recall: 0.8571
F1: 0.6462
Balanced accuracy: 0.7762

False negatives:
     imagenet_id                image         r    lambda         c  \
114           18     n01582220_magpie  0.160814  0.341802  0.228033   
30           111   n01930112_nematode  0.011978  0.494176  0.014975   
37           119  n01978455_rock